# Tema de Programare: Selecția Caracteristicilor Bazată pe Modele

Bine ai venit la tema de programare despre Selecția Caracteristicilor Bazată pe Modele!

Metodele de selecție a caracteristicilor bazate pe modele merg dincolo de filtrele statistice simple, antrenând modele de machine learning pentru a determina ce caracteristici sunt cu adevărat utile. În loc să măsoare relații statistice brute, aceste metode pun întrebarea: care caracteristici ajută acest model specific să obțină performanțe mai bune? Acest lucru duce la selecții aliniate mai bine cu algoritmul de învățare pe care intenționezi să îl folosești.

Metodele bazate pe regularizare, precum LASSO, anulează automat coeficienții caracteristicilor irelevante, în timp ce metodele bazate pe arbori expun importanțe ale caracteristicilor derivate direct din procesul de antrenare. Metodele wrapper precum RFE și Sequential Feature Selection evaluează subseturi reale de caracteristici prin antrenarea și scorarea modelelor, identificând combinațiile care funcționează bine împreună.

**Ce vei face în această temă:**

* **Selecție de Caracteristici cu LASSO** - Aplici regularizarea L1 pentru a identifica coeficienții nenuli și a extrage cele mai relevante caracteristici
* **Ajustarea lui Alpha** - Explorezi cum intensitatea regularizării controlează sparsitatea și numărul de caracteristici selectate
* **Importanță Bazată pe Arbori** - Extragi și compari importanța Gini (MDI) dintr-un clasificator Random Forest
* **Importanță prin Permutare** - Evaluezi importanța caracteristicilor pe date nevăzute pentru a evita biasul generat de cardinalitate mare
* **Eliminare Recursivă a Caracteristicilor** - Aplici RFE și verifici stabilitatea pe eșantioane bootstrap
* **Selecție Secvențială a Caracteristicilor** - Rulezi SFS forward și backward folosind scoruri validate încrucișat la fiecare pas
* **Selecție prin Ansamblu** - Agregi clasamentele din mai multe metode pentru o selecție finală mai robustă
* **Pipeline Bazat pe Modele** - Construiești un `Pipeline` sklearn complet cu selecție de caracteristici încorporată
* **Selecție Forward de la Zero** - Implementezi selecția greedy forward folosind acuratețe validată încrucișat

Să începem!


---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">SFATURI PENTRU EVALUAREA CU SUCCES A TEMEI:</h4>
* Toate celulele sunt blocate, cu excepția celor în care trebuie să trimiți soluțiile sau când este menționat explicit că poți interacționa cu ele.
* În fiecare celulă de exercițiu, caută comentariile `### ÎNCEPUT DE COD AICI ###` și `### SFÂRȘIT DE COD AICI ###`. Acestea îți arată unde să scrii codul soluției. **Nu adăuga și nu modifica niciun cod în afara acestor comentarii**.
* Poți adăuga celule noi pentru experimente, dar acestea vor fi ignorate de evaluator, așa că nu te baza pe celulele create de tine pentru codul soluției; folosește spațiile oferite în notebook.
* Evită folosirea variabilelor globale, cu excepția situațiilor absolut necesare. Evaluatorul îți testează codul într-un mediu izolat fără să ruleze toate celulele de la început. Drept urmare, variabilele globale pot să nu fie disponibile când este evaluată soluția ta. Variabilele globale care trebuie folosite vor fi definite cu MAJUSCULE.
* Pentru a trimite notebook-ul la evaluare, salvează-l mai întâi apăsând pe iconița 💾 din stânga sus a paginii, apoi apasă pe butonul `Submit assignment` din dreapta sus.
---


## Cuprins
- [Importuri](#setup)
- [1 - Selecția Caracteristicilor Bazată pe LASSO](#ex1)
    - **[Exercițiul 1 - lasso_feature_selection](#ex1)**
- [2 - Ajustarea Intensității Regularizării](#ex2)
    - **[Exercițiul 2 - tune_lasso_alpha](#ex2)**
- [3 - Importanța Caracteristicilor Bazată pe Arbori](#ex3)
    - **[Exercițiul 3 - get_tree_gini_importance](#ex3)**
- [4 - Importanță prin Permutare](#ex4)
    - **[Exercițiul 4 - get_permutation_importance](#ex4)**
- [5 - Eliminarea Recursivă a Caracteristicilor](#ex5)
    - **[Exercițiul 5 - apply_rfe](#ex5)**
- [6 - Stabilitatea RFE](#ex6)
    - **[Exercițiul 6 - check_rfe_stability](#ex6)**
- [7 - Selecția Secvențială a Caracteristicilor](#ex7)
    - **[Exercițiul 7 - apply_sequential_selection](#ex7)**
- [8 - Selecția prin Ansamblu a Caracteristicilor](#ex8)
    - **[Exercițiul 8 - ensemble_feature_selection](#ex8)**
- [9 - Pipeline Bazat pe Modele](#ex9)
    - **[Exercițiul 9 - create_model_based_pipeline](#ex9)**
- [Bonus - Selecție Forward de la Zero](#bonus)
    - **[Bonus - forward_selection_from_scratch](#bonus)**


<a name="setup"></a>
## Configurare


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import unittests
matplotlib.rcParams['figure.dpi'] = 100

from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Lasso, LassoCV, LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.feature_selection import (
    SelectFromModel, RFE, RFECV,
    SequentialFeatureSelector,
    f_classif, mutual_info_classif
)
from sklearn.inspection import permutation_importance
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score
from sklearn.utils import resample

np.random.seed(42)
print('Toate importurile au fost realizate cu succes!')


In [ ]:
# ── Set de date pentru clasificare ─────────────────────────────────────────
# 25 de caracteristici în total: 8 informative, 5 redundante, 12 zgomot
X_cls, y_cls = make_classification(
    n_samples=800, n_features=25, n_informative=8,
    n_redundant=5, n_repeated=0, n_clusters_per_class=2,
    random_state=42
)
FEATURE_NAMES_CLS = [f'feat_{i:02d}' for i in range(25)]

X_cls_tr, X_cls_te, y_cls_tr, y_cls_te = train_test_split(
    X_cls, y_cls, test_size=0.25, random_state=42
)

# Scalează pentru modelele liniare
scaler = StandardScaler()
X_cls_tr_sc = scaler.fit_transform(X_cls_tr)
X_cls_te_sc  = scaler.transform(X_cls_te)

# ── Set de date pentru regresie ────────────────────────────────────────────
# 20 de caracteristici: 6 informative
X_reg, y_reg = make_regression(
    n_samples=600, n_features=20, n_informative=6,
    noise=20, random_state=42
)
FEATURE_NAMES_REG = [f'rfeat_{i:02d}' for i in range(20)]

X_reg_tr, X_reg_te, y_reg_tr, y_reg_te = train_test_split(
    X_reg, y_reg, test_size=0.25, random_state=42
)

scaler_reg = StandardScaler()
X_reg_tr_sc = scaler_reg.fit_transform(X_reg_tr)
X_reg_te_sc  = scaler_reg.transform(X_reg_te)

print(f'Clasificare   -  train: {X_cls_tr.shape}, test: {X_cls_te.shape}')
print(f'Regresie      -  train: {X_reg_tr.shape}, test: {X_reg_te.shape}')


<a name="ex1"></a>
## Exercițiul 1  -  Selecția Caracteristicilor cu LASSO

Implementează `lasso_feature_selection` astfel încât:
1. Să încadreze un model `Lasso(alpha=alpha)` în `SelectFromModel(threshold='mean')`.
2. Să antreneze selectorul pe `(X_train, y_train)`.
3. Să returneze un **tuplu** `(selector, X_selected, coef_df)` unde:
   - `selector` este obiectul `SelectFromModel` antrenat.
   - `X_selected` este `X_train` transformat de selector.
   - `coef_df` este un `pd.DataFrame` cu coloanele `['feature', 'coefficient']`, sortat descrescător după `abs(coefficient)` și care conține **doar** caracteristicile selectate.


In [ ]:
def lasso_feature_selection(X_train, y_train, alpha, feature_names):
    """
    Aplică selecția caracteristicilor bazată pe LASSO prin SelectFromModel.

    Parameters
    ----------
    X_train : np.ndarray, shape (n_samples, n_features)
    y_train : np.ndarray, shape (n_samples,)
    alpha   : float   -  intensitatea regularizării pentru Lasso
    feature_names : list of str

    Returns
    -------
    selector   : SelectFromModel antrenat
    X_selected : np.ndarray, matricea de antrenare transformată
    coef_df    : pd.DataFrame, coloanele ['feature','coefficient'],
                 sortat descrescător după abs(coefficient), doar cele selectate
    """
    ### ÎNCEPUT DE COD AICI ###
    pass
    ### SFÂRȘIT DE COD AICI ###


In [ ]:
# Test 1  -  Selecția caracteristicilor cu LASSO
sel1, X_sel1, coef_df1 = lasso_feature_selection(X_reg_tr_sc, y_reg_tr, 0.1, FEATURE_NAMES_REG)
assert hasattr(sel1, 'get_support'), 'selector must be a SelectFromModel'
assert X_sel1.ndim == 2, 'X_selected must be 2-D'
assert X_sel1.shape[0] == X_reg_tr_sc.shape[0], 'row count must match'
assert X_sel1.shape[1] == sel1.get_support().sum(), 'column count must match selector'
assert list(coef_df1.columns) == ['feature', 'coefficient'], 'wrong columns'
assert len(coef_df1) == X_sel1.shape[1], 'coef_df must have one row per selected feature'
# Verifică sortarea descrescătoare după |coef|
abs_coefs = coef_df1['coefficient'].abs().values
assert all(abs_coefs[i] >= abs_coefs[i+1] for i in range(len(abs_coefs)-1)), 'must be sorted desc'
print(f'Exercițiul 1 a fost rezolvat corect! Au fost selectate {X_sel1.shape[1]}/{X_reg_tr_sc.shape[1]} caracteristici.')
print(coef_df1.head())


In [ ]:
unittests.exercise_1(lasso_feature_selection)

<a name="ex2"></a>
## Exercițiul 2  -  Ajustarea lui Alpha pentru LASSO

Implementează `tune_lasso_alpha` astfel încât:
1. Să itereze prin fiecare valoare din `alphas`.
2. Să antreneze un `Lasso(alpha=a)` pe `(X_train, y_train)`.
3. Să numere coeficienții nenuli.
4. Să returneze un `pd.DataFrame` cu coloanele `['alpha', 'n_selected']`.


In [ ]:
def tune_lasso_alpha(X_train, y_train, alphas):
    """
    Returnează un DataFrame care arată cum se schimbă
    numărul de caracteristici selectate în funcție de alpha.

    Parameters
    ----------
    X_train : np.ndarray
    y_train : np.ndarray
    alphas  : list of float

    Returns
    -------
    pd.DataFrame cu coloanele ['alpha', 'n_selected']
    """
    ### ÎNCEPUT DE COD AICI ###
    pass
    ### SFÂRȘIT DE COD AICI ###


In [ ]:
# Test 2  -  Ajustarea lui alpha pentru LASSO
alphas_grid = [0.001, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0]
alpha_df = tune_lasso_alpha(X_reg_tr_sc, y_reg_tr, alphas_grid)
assert list(alpha_df.columns) == ['alpha', 'n_selected'], 'wrong columns'
assert len(alpha_df) == len(alphas_grid), 'must have one row per alpha'
assert alpha_df['n_selected'].iloc[0] >= alpha_df['n_selected'].iloc[-1], \
    'smaller alpha should retain >= features than larger alpha'
print('Exercițiul 2 a fost rezolvat corect!')
# Grafic
plt.semilogx(alpha_df['alpha'], alpha_df['n_selected'], marker='o')
plt.xlabel('Alpha'); plt.ylabel('Caracteristici păstrate'); plt.title('LASSO: Alpha vs Caracteristici')
plt.grid(True); plt.tight_layout(); plt.show()


In [ ]:
unittests.exercise_2(tune_lasso_alpha)

<a name="ex3"></a>
## Exercițiul 3  -  Importanța Caracteristicilor Bazată pe Arbori (Gini)

Implementează `get_tree_gini_importance` astfel încât:
1. Să antreneze un `RandomForestClassifier(n_estimators=200, random_state=42)`.
2. Să returneze un `pd.DataFrame` cu coloanele `['feature', 'importance']`,
   sortat **descrescător** după `importance`.


In [ ]:
def get_tree_gini_importance(X_train, y_train, feature_names):
    """
    Calculează importanțele bazate pe Mean Decrease in Impurity (Gini).

    Returns
    -------
    pd.DataFrame cu coloanele ['feature', 'importance'], sortat descrescător.
    Returnează și clasificatorul RandomForest antrenat ca al doilea element.
    """
    ### ÎNCEPUT DE COD AICI ###
    pass
    ### SFÂRȘIT DE COD AICI ###


In [ ]:
# Test 3  -  Importanța Gini din arbori
gini_df, rf_model = get_tree_gini_importance(X_cls_tr, y_cls_tr, FEATURE_NAMES_CLS)
assert list(gini_df.columns) == ['feature', 'importance'], 'wrong columns'
assert len(gini_df) == len(FEATURE_NAMES_CLS), 'must have one row per feature'
assert abs(gini_df['importance'].sum() - 1.0) < 1e-6, 'importances must sum to 1'
assert gini_df['importance'].iloc[0] >= gini_df['importance'].iloc[-1], 'must be sorted desc'
print(f'Exercițiul 3 a fost rezolvat corect! Cea mai importantă caracteristică: {gini_df.iloc[0]["feature"]} ({gini_df.iloc[0]["importance"]:.4f})')
# Reprezintă top 10
top10 = gini_df.head(10)
plt.figure(figsize=(8,4))
plt.barh(top10['feature'][::-1], top10['importance'][::-1], color='steelblue')
plt.xlabel('Importanță MDI'); plt.title('Top 10 importanțe Gini'); plt.tight_layout(); plt.show()


In [ ]:
unittests.exercise_3(get_tree_gini_importance)

<a name="ex4"></a>
## Exercițiul 4  -  Importanță prin Permutare

Implementează `get_permutation_importance` astfel încât:
1. Să apeleze `permutation_importance(model, X_val, y_val, n_repeats=30, random_state=42, n_jobs=-1)`.
2. Să returneze un `pd.DataFrame` cu coloanele `['feature', 'importance_mean', 'importance_std']`,
   sortat **descrescător** după `importance_mean`.


In [ ]:
def get_permutation_importance(model, X_val, y_val, feature_names):
    """
    Calculează importanța prin permutare pe datele de validare.

    Parameters
    ----------
    model        : estimator sklearn antrenat
    X_val        : np.ndarray  -  caracteristici de validare
    y_val        : np.ndarray  -  etichete de validare
    feature_names: list of str

    Returns
    -------
    pd.DataFrame cu coloanele ['feature','importance_mean','importance_std'],
    sortat descrescător după importance_mean.
    """
    ### ÎNCEPUT DE COD AICI ###
    pass
    ### SFÂRȘIT DE COD AICI ###


In [ ]:
# Test 4  -  Importanță prin permutare
perm_df = get_permutation_importance(rf_model, X_cls_te, y_cls_te, FEATURE_NAMES_CLS)
assert list(perm_df.columns) == ['feature','importance_mean','importance_std'], 'wrong columns'
assert len(perm_df) == len(FEATURE_NAMES_CLS), 'must have one row per feature'
assert perm_df['importance_mean'].iloc[0] >= perm_df['importance_mean'].iloc[-1], 'must be sorted desc'
print('Exercițiul 4 a fost rezolvat corect!')
# Comparație una lângă alta
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
gini_df.head(10).plot.barh(x='feature', y='importance', ax=ax1, color='steelblue', legend=False)
ax1.set_title('Importanță Gini'); ax1.invert_yaxis()
perm_df.head(10).plot.barh(x='feature', y='importance_mean', ax=ax2, color='darkorange', legend=False)
ax2.set_title('Importanță prin permutare'); ax2.invert_yaxis()
plt.tight_layout(); plt.show()


In [ ]:
unittests.exercise_4(get_permutation_importance)

<a name="ex5"></a>
## Exercițiul 5  -  Eliminarea Recursivă a Caracteristicilor

Implementează `apply_rfe` astfel încât:
1. Să creeze `RFE(estimator=estimator, n_features_to_select=n_features, step=1)`.
2. Să îl antreneze pe `(X_train, y_train)`.
3. Să returneze un **tuplu** `(rfe, X_selected, ranking_df)` unde:
   - `rfe` este obiectul RFE antrenat.
   - `X_selected` este `X_train` transformat prin `rfe`.
   - `ranking_df` este un `pd.DataFrame` cu coloanele `['feature', 'rank']`,
     sortat crescător după `rank` (rangul 1 = selectat).


In [ ]:
def apply_rfe(X_train, y_train, estimator, n_features, feature_names):
    """
    Aplică Eliminarea Recursivă a Caracteristicilor.

    Returns
    -------
    rfe         : RFE antrenat
    X_selected  : np.ndarray
    ranking_df  : pd.DataFrame cu coloanele ['feature','rank'], sortat crescător după rank
    """
    ### ÎNCEPUT DE COD AICI ###
    pass
    ### SFÂRȘIT DE COD AICI ###


In [ ]:
# Test 5  -  RFE
from sklearn.linear_model import LogisticRegression
lr_est = LogisticRegression(max_iter=2000, random_state=42)
rfe5, X_rfe, rank_df = apply_rfe(X_cls_tr_sc, y_cls_tr, lr_est, 8, FEATURE_NAMES_CLS)
assert hasattr(rfe5, 'support_'), 'must return fitted RFE'
assert X_rfe.shape == (X_cls_tr_sc.shape[0], 8), f'expected shape ({X_cls_tr_sc.shape[0]}, 8), got {X_rfe.shape}'
assert list(rank_df.columns) == ['feature', 'rank'], 'wrong columns'
assert rank_df['rank'].iloc[0] == 1, 'first row must be rank 1 (selected)'
assert rfe5.support_.sum() == 8, 'must select exactly 8 features'
print(f'Exercițiul 5 a fost rezolvat corect! Caracteristici selectate: {list(rank_df[rank_df["rank"]==1]["feature"])}')


In [ ]:
unittests.exercise_5(apply_rfe)

<a name="ex6"></a>
## Exercițiul 6  -  Verificarea Stabilității RFE

Implementează `check_rfe_stability` astfel încât:
1. Pentru fiecare din cele `n_splits` resamplări bootstrap ale lui `(X, y)`, să ruleze `RFE(estimator, n_features, step=1)`.
   Folosește `resample(X, y, random_state=i)` pentru `i` în intervalul `n_splits`.
2. Să colecteze setul de indici ai caracteristicilor selectate din fiecare rulare.
3. Să calculeze **media similarității Jaccard pe perechi** pentru toate perechile de rulări.
4. Să returneze `(mean_jaccard, selected_sets)` ca float și listă de seturi.


In [ ]:
def check_rfe_stability(X, y, estimator, n_features, n_splits=20):
    """
    Măsoară stabilitatea selecției RFE prin resamplare bootstrap.

    Parameters
    ----------
    X, y        : date
    estimator   : estimator sklearn (va fi clonat în interior)
    n_features  : int  -  numărul țintă de caracteristici
    n_splits    : int  -  numărul de rulări bootstrap

    Returns
    -------
    mean_jaccard   : float
    selected_sets  : listă de seturi de int (indicii caracteristicilor selectate)
    """
    ### ÎNCEPUT DE COD AICI ###
    pass
    ### SFÂRȘIT DE COD AICI ###


In [ ]:
# Test 6  -  Stabilitatea RFE
from sklearn.linear_model import LogisticRegression
lr_est6 = LogisticRegression(max_iter=2000, random_state=42)
jac, sets = check_rfe_stability(X_cls_tr_sc, y_cls_tr, lr_est6, 8, n_splits=10)
assert 0.0 <= jac <= 1.0, 'Jaccard must be in [0, 1]'
assert len(sets) == 10, 'must have one set per split'
assert all(isinstance(s, set) for s in sets), 'each element must be a set'
print(f'Exercițiul 6 a fost rezolvat corect! Stabilitate Jaccard medie: {jac:.3f}')


In [ ]:
unittests.exercise_6(check_rfe_stability)

<a name="ex7"></a>
## Exercițiul 7  -  Selecția Secvențială a Caracteristicilor

Implementează `apply_sequential_selection` astfel încât:
1. Să creeze un `SequentialFeatureSelector(estimator, n_features_to_select=n_features,
   direction=direction, scoring='accuracy', cv=3)`.
2. Să îl antreneze pe `(X_train, y_train)`.
3. Să returneze obiectul `sfs` antrenat și lista numelor caracteristicilor selectate.

> **Pont:** `direction` poate fi `'forward'` sau `'backward'`.


In [ ]:
def apply_sequential_selection(X_train, y_train, estimator,
                                n_features, feature_names,
                                direction='forward'):
    """
    Aplică Selecția Secvențială a Caracteristicilor (forward sau backward).

    Returns
    -------
    sfs              : SequentialFeatureSelector antrenat
    selected_features: listă cu șirurile numelor caracteristicilor selectate
    """
    ### ÎNCEPUT DE COD AICI ###
    pass
    ### SFÂRȘIT DE COD AICI ###


In [ ]:
# Test 7  -  Selecția Secvențială a Caracteristicilor (forward)
from sklearn.linear_model import LogisticRegression
lr_est7 = LogisticRegression(max_iter=2000, random_state=42)
sfs7, sfs_feat = apply_sequential_selection(
    X_cls_tr_sc, y_cls_tr, lr_est7, 6, FEATURE_NAMES_CLS, direction='forward'
)
assert hasattr(sfs7, 'get_support'), 'must return fitted SFS'
assert len(sfs_feat) == 6, f'expected 6 features, got {len(sfs_feat)}'
assert all(f in FEATURE_NAMES_CLS for f in sfs_feat), 'all names must be valid'
print(f'Exercițiul 7 a fost rezolvat corect! Selectate: {sfs_feat}')


In [ ]:
unittests.exercise_7(apply_sequential_selection)

<a name="ex8"></a>
## Exercițiul 8  -  Selecția prin Ansamblu a Caracteristicilor

Implementează `ensemble_feature_selection` astfel încât:
1. Să ruleze **trei** metode de selecție și să colecteze **un rang per caracteristică** (1 = cea mai importantă) din fiecare:
   - `f_classif` (filtru)
   - `RandomForestClassifier(n_estimators=100, random_state=42)` (importanță Gini)
   - `LogisticRegression(penalty='l1', C=0.5, solver='liblinear', max_iter=1000)` (coeficienți L1)
2. Să calculeze media celor trei ranguri.
3. Să returneze un `pd.DataFrame` cu coloanele `['feature','rank_f','rank_rf','rank_l1','mean_rank']`,
   sortat crescător după `mean_rank`, și o listă cu numele caracteristicilor din **top-k**.


In [ ]:
def ensemble_feature_selection(X_train, y_train, feature_names, k=10):
    """
    Agregă clasamentele caracteristicilor din f_classif, RF Gini și regresie logistică L1.

    Returns
    -------
    rank_df  : pd.DataFrame cu coloanele ['feature','rank_f','rank_rf','rank_l1','mean_rank']
               sortat crescător după mean_rank
    top_k    : list of str  -  cele k nume de caracteristici cu mean_rank minim
    """
    ### ÎNCEPUT DE COD AICI ###
    pass
    ### SFÂRȘIT DE COD AICI ###


In [ ]:
# Test 8  -  Selecția prin ansamblu a caracteristicilor
ens_df, top8 = ensemble_feature_selection(X_cls_tr_sc, y_cls_tr, FEATURE_NAMES_CLS, k=8)
assert list(ens_df.columns) == ['feature','rank_f','rank_rf','rank_l1','mean_rank'], 'wrong columns'
assert len(ens_df) == len(FEATURE_NAMES_CLS), 'must have one row per feature'
assert len(top8) == 8, 'top_k must have k elements'
assert ens_df['mean_rank'].iloc[0] <= ens_df['mean_rank'].iloc[-1], 'must be sorted asc'
print(f'Exercițiul 8 a fost rezolvat corect! Top-8 după rangul de ansamblu: {top8}')
print(ens_df.head(8))


In [ ]:
unittests.exercise_8(ensemble_feature_selection)

<a name="ex9"></a>
## Exercițiul 9  -  Construirea unui Pipeline Bazat pe Modele

Implementează `create_model_based_pipeline` astfel încât:
1. Să accepte `method` ∈ `{'lasso', 'rf'}` și `k` (numărul de caracteristici selectate).
2. Să construiască și să returneze un `Pipeline` sklearn cu următorii pași:
   - `'scaler'`: `StandardScaler()`
   - `'selector'`:
     - Dacă `method == 'lasso'`: `SelectFromModel(Lasso(alpha=0.01), max_features=k, threshold=-np.inf)`
     - Dacă `method == 'rf'`:   `SelectFromModel(RandomForestClassifier(n_estimators=100, random_state=42), max_features=k, threshold=-np.inf)`
   - `'model'`: `LogisticRegression(max_iter=1000)`
3. Funcția trebuie să ridice `ValueError` pentru valori necunoscute ale lui `method`.


In [ ]:
def create_model_based_pipeline(method, k):
    """
    Construiește un Pipeline: StandardScaler -> SelectFromModel -> LogisticRegression.

    Parameters
    ----------
    method : str   -  'lasso' sau 'rf'
    k      : int   -  numărul maxim de caracteristici de păstrat (max_features)

    Returns
    -------
    sklearn Pipeline

    Raises
    ------
    ValueError pentru o metodă necunoscută
    """
    ### ÎNCEPUT DE COD AICI ###
    pass
    ### SFÂRȘIT DE COD AICI ###


In [ ]:
# Test 9  -  Pipeline bazat pe modele
import sklearn

for method in ['lasso', 'rf']:
    pipe = create_model_based_pipeline(method, k=8)
    assert hasattr(pipe, 'fit'), f'{method} pipeline must have fit()'
    assert list(pipe.named_steps.keys()) == ['scaler','selector','model'], \
        'pipeline must have exactly: scaler, selector, model'
    scores = cross_val_score(pipe, X_cls, y_cls, cv=5, scoring='accuracy')
    print(f'  method={method} -> acuratețe în 5 fold-uri: {scores.mean():.3f} ± {scores.std():.3f}')

try:
    create_model_based_pipeline('unknown', 5)
    assert False, 'Should have raised ValueError'
except ValueError:
    pass
print('Exercițiul 9 a fost rezolvat corect!')


In [ ]:
unittests.exercise_9(create_model_based_pipeline)

<a name="bonus"></a>
## Exercițiu Bonus  -  Selecție Forward de la Zero

Implementează `forward_selection_from_scratch` astfel încât:
1. Să pornească de la un set gol de indici ai caracteristicilor selectate.
2. La fiecare pas, să încerce adăugarea fiecărei caracteristici **rămase** la setul curent.
3. Să evalueze scorul de validare încrucișată (acuratețe) cu `LogisticRegression(max_iter=1000)` și `cv=3`.
4. Să adauge permanent caracteristica care oferă **cel mai mare** scor CV la acel pas.
5. Să se oprească atunci când au fost selectate `k` caracteristici.
6. Să returneze `(selected_indices, score_history)` unde:
   - `selected_indices` este o listă de numere întregi (indicii coloanelor în ordinea selecției).
   - `score_history` este o listă de valori float (cel mai bun scor CV la fiecare pas).


In [ ]:
def forward_selection_from_scratch(X, y, k):
    """
    Selecție greedy forward a caracteristicilor folosind acuratețe validată încrucișat.

    Parameters
    ----------
    X : np.ndarray, shape (n_samples, n_features)
    y : np.ndarray, shape (n_samples,)
    k : int  -  numărul de caracteristici de selectat

    Returns
    -------
    selected_indices : list of int    -  indicii caracteristicilor selectate
    score_history    : list of float  -  cel mai bun scor CV la fiecare pas
    """
    ### ÎNCEPUT DE COD AICI ###
    pass
    ### SFÂRȘIT DE COD AICI ###


In [ ]:
# Test bonus  -  Selecție forward de la zero
sel_idx, scores_hist = forward_selection_from_scratch(X_cls_tr_sc, y_cls_tr, k=8)
assert len(sel_idx) == 8, 'must select exactly k features'
assert len(scores_hist) == 8, 'score_history must have k entries'
assert all(0 <= s <= 1 for s in scores_hist), 'scores must be in [0,1]'
assert len(set(sel_idx)) == 8, 'all selected indices must be unique'
print(f'Bonus rezolvat! Indici selectați: {sel_idx}')
plt.plot(range(1, 9), scores_hist, marker='o')
plt.xlabel('Numărul de caracteristici'); plt.ylabel('Acuratețe CV')
plt.title('Selecție Forward: Acuratețe CV vs Numărul de Caracteristici')
plt.grid(True); plt.tight_layout(); plt.show()


In [ ]:
unittests.exercise_10(forward_selection_from_scratch)